In [1]:
%pip -q install -r requirements.txt
# For local .nemo models, run once, then restart the kernel:
# %pip install -r requirements-nemo.txt
import os
import shutil
import sys
from pathlib import Path
from dotenv import load_dotenv

# Read private Hugging Face credentials before importing any model libraries.
load_dotenv(dotenv_path=Path.cwd() / ".env", override=False)

# Also works when VS Code starts the kernel without sourcing local-env.sh.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("XDG_CACHE_HOME", str(Path.cwd() / ".cache"))
os.environ.setdefault("HF_HOME", str(Path.cwd() / ".cache" / "huggingface"))
os.environ.setdefault("NEMO_CACHE_DIR", str(Path.cwd() / ".cache" / "nemo"))
if not shutil.which("ffmpeg"):
    raise RuntimeError("FFmpeg not found. Run bash setup-local.sh in the project terminal first.")
print("Dependencies and FFmpeg ready.")


Note: you may need to restart the kernel to use updated packages.
Dependencies and FFmpeg ready.


In [ ]:
import gc
import re
import time
import torch
import whisper
import pandas as pd
from IPython.display import display

AUDIO_DIR = "./selected_audio"
OUTPUT_CSV = "./model_comparison.csv"
LANGUAGE = "en"
BATCH_STATE = "default"  # Keyword prompt: "decision", "reason", or "default" (both).

# Edit this array to choose models. Each name must be unique (used in CSV columns).
MODELS = [
    {"name": "tiny", "backend": "whisper", "model": "tiny"},
    {"name": "base", "backend": "whisper", "model": "base"},
    {"name": "small", "backend": "whisper", "model": "small"},
    # Download a published NeMo checkpoint automatically, then run locally:
    {"name": "parakeet_v3", "backend": "nemo", "model": "nvidia/parakeet-tdt-0.6b-v3", "pretrained": True},
    # Local NeMo ASR checkpoint (e.g. Parakeet). Replace with your actual file path:
    {"name": "parakeet_finetuned_museum", "backend": "nemo", "model": "~/Downloads/parakeet_adapter_museum_best.nemo"},
    # Private NeMo checkpoint on Hugging Face (uses HF_TOKEN from .env):
    {"name": "parakeet_finetuned", "backend": "nemo", "model": "norillalearning/parakeet_finetuned", "pretrained": True, "filename": "parakeet_fullft_finetuned.nemo"},
    # Transformers-format Whisper repos use backend="huggingface".
    # If the fine-tuned repo has no tokenizer/processor, add its matching base:
    # {"name": "finetuned", "backend": "huggingface", "model": "norillalearning/YOUR_MODEL_REPO", "processor": "openai/whisper-small"},
]

MODEL_CACHE = Path(".cache").resolve()

def get_model(spec):
    print(f"Loading {spec['name']}: {spec['backend']} / {spec['model']}...")
    if spec["backend"] == "whisper":
        return whisper.load_model(spec["model"], device="cpu", download_root=str(MODEL_CACHE / "whisper"))
    if spec["backend"] == "huggingface":
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
        processor = AutoProcessor.from_pretrained(
            spec.get("processor", spec["model"]), cache_dir=str(MODEL_CACHE / "huggingface"),
        )
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            spec["model"], cache_dir=str(MODEL_CACHE / "huggingface"), dtype=torch.float32,
        )
        if model.config.model_type != "whisper":
            raise ValueError("The huggingface backend expects a Whisper speech-to-text checkpoint.")
        return pipeline(
            "automatic-speech-recognition", model=model,
            tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
            device=-1,
        )
    if spec["backend"] == "nemo":
        if not spec.get("pretrained", False):
            checkpoint = Path(spec["model"]).expanduser().resolve()
            if checkpoint.suffix.lower() != ".nemo":
                raise ValueError("Use a local .nemo file, or set pretrained=True for a NeMo model ID.")
            if not checkpoint.is_file():
                raise FileNotFoundError(f"NeMo checkpoint not found: {checkpoint}")
        try:
            from nemo.collections.asr.models import ASRModel
        except ImportError as exc:
            raise ImportError(
                "NeMo ASR could not be imported. Run %pip install -r requirements-nemo.txt "
                f"and restart the kernel. Original error: {exc}"
            ) from exc
        if spec.get("pretrained", False):
            if spec.get("filename"):
                from huggingface_hub import hf_hub_download
                # Use the requested checkpoint, including names that differ from the repo.
                # Hugging Face reads HF_TOKEN from the environment loaded in cell 1.
                checkpoint = Path(hf_hub_download(repo_id=spec["model"], filename=spec["filename"]))
            else:
                checkpoint = Path(ASRModel.from_pretrained(
                    model_name=spec["model"], return_model_file=True,
                ))
        # Some adapter-tuned exports still name the base encoder in their config.
        # Restore its registered adapter class so the saved adapter weights stay active.
        config = ASRModel.restore_from(restore_path=str(checkpoint), return_config=True)
        if config.get("adapters") and config.get("encoder", {}).get("_target_"):
            from nemo.core.classes.mixins import adapter_mixins
            from omegaconf import open_dict
            adapter_info = adapter_mixins.get_registered_adapter(config.encoder._target_)
            if adapter_info is not None and config.encoder._target_ != adapter_info.adapter_class_path:
                with open_dict(config):
                    config.encoder._target_ = adapter_info.adapter_class_path
                print(f"Restoring adapter-compatible encoder: {config.encoder._target_}")
        model = ASRModel.restore_from(
            restore_path=str(checkpoint), map_location=torch.device("cpu"),
            override_config_path=config, strict=True,
        )
        model = model.to(device="cpu", dtype=torch.float32).eval()
        sample_rate = model.cfg.get("preprocessor", {}).get("sample_rate", 16000)
        if sample_rate != 16000:
            raise ValueError(f"This comparison uses 16 kHz audio; the NeMo checkpoint expects {sample_rate} Hz.")
        print("NeMo uses checkpoint decoding defaults; Whisper keyword prompt/language settings are not applied.")
        return model
    raise ValueError(f"Unsupported backend: {spec['backend']}")

def transcribe_audio(model, spec, audio, prompt):
    # All models use the same 16 kHz audio on CPU/float32. Whisper backends share a prompt.
    if spec["backend"] == "whisper":
        return model.transcribe(
            audio, initial_prompt=prompt, language=LANGUAGE,
            fp16=False, temperature=0.0, beam_size=5,
        )["text"].strip()
    if spec["backend"] == "nemo":
        # In-memory audio avoids re-decoding the WAV inside the measured call.
        with torch.inference_mode():
            results = model.transcribe(
                audio=[audio.copy()], batch_size=1, num_workers=0,
                return_hypotheses=False, verbose=False,
            )
        # NeMo versions/models return strings, Hypothesis objects, or (best, alternatives).
        if isinstance(results, tuple):
            results = results[0]
        if not isinstance(results, list) or len(results) != 1:
            raise ValueError("Expected one transcription from NeMo for one audio file.")
        result = results[0]
        text = result if isinstance(result, str) else getattr(result, "text", None)
        if not isinstance(text, str):
            raise TypeError(f"Unsupported NeMo transcription result: {type(result).__name__}")
        return text.strip()
    kwargs = {"num_beams": 5, "temperature": 0.0, "do_sample": False}
    if getattr(model.model.generation_config, "is_multilingual", True):
        kwargs.update(language=LANGUAGE, task="transcribe")
    if prompt:
        kwargs["prompt_ids"] = model.tokenizer.get_prompt_ids(prompt, return_tensors="pt")
    return model(
        {"array": audio.copy(), "sampling_rate": 16000}, generate_kwargs=kwargs,
        return_timestamps=len(audio) > 30 * 16000,
    )["text"].strip()

def normalize_text(text):
    text = str(text).lower().strip()
    return re.sub(r"[^\w\s]", "", text)

print("Models configured:", [spec["name"] for spec in MODELS])
print("Input:", Path(AUDIO_DIR).resolve())
print("Output:", Path(OUTPUT_CSV).resolve())


In [3]:
INTENT_CONFIG = {
    "decision": {
        "tower.decision.right": [["right"], ["2"], ["two"], ["second"]],
        "tower.decision.left": [["left"], ["1"], ["number", "one"], ["first"]],
        "tower.decision.same": [["both"], ["together"], ["same", "time"], ["one", "and", "two"]]
    },
    "reason": {
        "tower.reason.tall": [["tall"], ["taller"], ["big"], ["bigger"], ["high"], ["higher"]],
        "tower.reason.weight": [["weight"], ["weighs"], ["top"], ["heavy"], ["heavier"]],
        "tower.reason.thinnerbase": [["thin"], ["thinner"], ["small"], ["smaller"], ["base"], ["bottom"]],
        "tower.reason.notsymmetrical": [["symmetry"], ["symmetric"], ["symmetrical"], ["both", "sides"], ["not", "the", "same"], ["asymmetrical"], ["asymmetric"], ["not", "same"]]
    }
}

def determine_intent(transcript, state):
    """Implementation of the 3-step intent matching logic."""
    if not transcript or str(transcript).strip() == "":
        return "default.noaudio"

    # Step 1: Clean text
    clean = normalize_text(transcript)
    words = clean.split()

    # Step 2: Get intents for current state
    state_intents = INTENT_CONFIG.get(state, {})

    # Step 3: Keyword group matching
    for intent_name, groups in state_intents.items():
        for group in groups:
            # Match if EVERY keyword in group is found as a whole word
            if all(any(re.fullmatch(re.escape(k), w) for w in words) for k in group):
                return intent_name

    return "default.unknown" # Or noaudio if you prefer

In [ ]:
KEYWORDS = {
    "reason": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric"
    ],
    "decision": [
        "right", "two", "left", "one", "both", "together", "same time",
    ],
    "default": [
        "tall", "taller", "big", "high", "higher", "weight", "weighs", "top",
        "heavy", "heavier", "thin", "thinner", "small", "smaller",
        "symmetric", "symmetrical", "both sides", "asymmetrical", "asymmetric",
        "right", "two", "left", "one", "both", "together", "same time",
    ],
}


def run_model_comparison(models, output_csv, state):
    if not models:
        raise ValueError("Add at least one entry to MODELS.")
    names = [spec["name"] for spec in models]
    if any(not name.strip() for name in names) or len(set(names)) != len(names):
        raise ValueError("Every model needs a nonempty, unique name.")
    for spec in models:
        if spec["backend"] not in {"whisper", "huggingface", "nemo"} or not spec["model"]:
            raise ValueError(f"Invalid model configuration: {spec}")
    if state not in KEYWORDS:
        raise ValueError("BATCH_STATE must be decision, reason, or default.")

    folder = Path(AUDIO_DIR)
    if not folder.is_dir():
        raise FileNotFoundError(f"Audio folder not found: {folder.resolve()}")
    files = sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() == ".wav")
    if not files:
        raise ValueError(f"No WAV files found in {folder.resolve()}")

    # Folder contents determine the rows; no input CSV or existing transcript is needed.
    rows, audio_inputs, audio_errors = [], {}, {}
    for path in files:
        row = {"fileName": path.name, "audio duration [s]": None}
        try:
            audio = whisper.load_audio(str(path))  # Decode once at 16 kHz, outside the timer.
            audio_inputs[path.name] = audio
            row["audio duration [s]"] = round(len(audio) / 16000, 3)
        except Exception as exc:
            audio_errors[path.name] = str(exc)
        rows.append(row)
    df = pd.DataFrame(rows)
    fields = ["model", "transcription", "intent [decision]", "intent [reason]", "processing time [ms]", "status", "error"]
    for spec in models:
        for field in fields:
            df[f"{spec['name']} | {field}"] = None
        model_id = f"{spec['backend']}:{spec['model']}"
        if spec.get("filename"):
            model_id += "/" + spec["filename"]
        df[f"{spec['name']} | model"] = model_id

    output_path = Path(output_csv)
    if output_path.resolve() in {p.resolve() for p in Path('.').glob('*_audio_log.csv')}:
        raise ValueError("Use a separate OUTPUT_CSV, not an original audio log.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    prompt = "Try to focus specifically on these words as you listen to the audio: " + ", ".join(KEYWORDS[state])
    print(f"Comparing {len(models)} models on {len(files)} WAV files. CPU / float32.")
    print("Processing time excludes download, model loading, audio decoding and intent classification.")
    if any(spec["backend"] == "nemo" for spec in models):
        print("NeMo uses checkpoint decoding defaults, without Whisper keyword prompts.")

    for spec in models:
        prefix = spec["name"] + " | "
        model = None
        try:
            model = get_model(spec)  # One model in memory at a time; loading is not timed.
            for idx, row in df.iterrows():
                file_name = row["fileName"]
                try:
                    if file_name in audio_errors:
                        raise ValueError(audio_errors[file_name])
                    start = time.perf_counter()
                    text = transcribe_audio(model, spec, audio_inputs[file_name], prompt)
                    elapsed_ms = (time.perf_counter() - start) * 1000
                    df.at[idx, prefix + "transcription"] = text
                    df.at[idx, prefix + "intent [decision]"] = determine_intent(text, "decision")
                    df.at[idx, prefix + "intent [reason]"] = determine_intent(text, "reason")
                    df.at[idx, prefix + "processing time [ms]"] = round(elapsed_ms, 2)
                    df.at[idx, prefix + "status"] = "ok"
                    print(f"{spec['name']} / {file_name}: {text!r} ({elapsed_ms:.0f} ms)")
                except Exception as exc:
                    df.at[idx, prefix + "status"] = "error"
                    df.at[idx, prefix + "error"] = str(exc)
                    print(f"ERROR {spec['name']} / {file_name}: {exc}")
        except Exception as exc:
            df[prefix + "status"] = "load_error"
            df[prefix + "error"] = str(exc)
            print(f"ERROR loading {spec['name']}: {exc}")
        finally:
            del model
            gc.collect()
            df.to_csv(output_path, index=False)  # Preserve other models' results on failure.

    print(f"Saved {len(df)} rows to {output_path.resolve()}")
    display(df)
    return df

# Each run creates a fresh comparison of the current model list and selected WAVs.
comparison_df = run_model_comparison(MODELS, OUTPUT_CSV, BATCH_STATE)


In [ ]:
# Quick timing summary. Raw transcription and both intents are in OUTPUT_CSV.
summary = []
for spec in MODELS:
    prefix = spec["name"] + " | "
    successful = comparison_df[prefix + "status"].eq("ok")
    timings = pd.to_numeric(comparison_df.loc[successful, prefix + "processing time [ms]"])
    summary.append({
        "model": spec["name"],
        "successful files": int(successful.sum()),
        "failed files": int((~successful).sum()),
        "mean processing time [ms]": round(timings.mean(), 2),
    })
display(pd.DataFrame(summary))
print("Timing compares these local runs; it is not a transcription accuracy score.")
